<a href="https://colab.research.google.com/github/adryann-olivatti/Damicore_IC_notebook1_jose/blob/main/Resumo_2_inicia%C3%A7%C3%A3o_cientifica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Método FS-OPA: Documentação e Implementação Completa e Detalhada

Este notebook contém a implementação prática do método **FS-OPA** (*Feature Sensitivity through criterion-based resampling from Phylogram Analysis*) acompanhada da **explicação teórica, matemática e técnica aprofundada**.

## 1. O que são as "Variáveis" no Dataset?

Em termos práticos, **variáveis** (ou atributos) são as colunas da planilha de dados (`Base2FABC.xlsx`). Cada linha representa um talhão/lavoura, e cada coluna representa uma característica medida:

- **Variável Alvo (Dependente):** `Produtividade [kg/ha]` (o resultado final que queremos explicar).
- **Variáveis de Solo e Nutrição:** pH, Fósforo, Potássio, Matéria Orgânica, Saturação por Bases, etc.
- **Variáveis Climáticas:** Precipitação, Temperatura, Graus Dias Acumulados (GDA), Dias de Estresse Hídrico.
- **Variáveis de Manejo Agronômico:** Data de Semeadura, Espaçamento, Doses de Fertilizantes (N, P, K), Cultivar.
- **Variáveis Geográficas/Topográficas:** Altitude Máxima, Altitude Mínima, Declividade.

## PILAR 1: FUNDAMENTAÇÃO TEÓRICA APROFUNDADA DO FS-OPA

O método **FS-OPA** é uma abordagem não paramétrica e independente de modelo (*model-agnostic*) criada para medir a sensibilidade/relevância de atributos em relação a um atributo-alvo.

### 1.1 O Princípio da Reamostragem Baseada em Critérios (*Criterion-Based Resampling*)
Ao recortar o dataset em pares opostos:
* **$B_{2C}$ (Best 50%) vs. $W_{2C}$ (Worst 50%):** Avalia tendências gerais de alto vs. baixo desempenho.
* **$B_{4C}$ (Best 25%) vs. $W_{4C}$ (Worst 25%):** Isola os quartis extremos, eliminando a "zona cinzenta" intermediária.
* **$B_{8C}$ (Best 12.5%) vs. $W_{8C}$ (Worst 12.5%):** Isola as super-lavouras contra as lavouras sob estresse severo ou falhas graves de manejo.

### 1.2 Teoria da Informação e Complexidade de Kolmogorov
Baseia-se na **Complexidade Algorítmica de Kolmogorov $K(x)$**, aproximada pelo tamanho do arquivo comprimido por um algoritmo sem perdas (ex: `gzip`). Isso mede a quantidade de **Entropia de Shannon** do sinal de dados do atributo.

### 1.3 Agrupamento Filogenético
* As **folhas** do filograma são os atributos agronômicos.
* Os **ramos** representam a distância de informação entre eles.
* Se a variável $X$ e a Produtividade estão no mesmo clado (ramo fechado), elas compartilham padrão de variação e estrutura de informação equivalentes.

## BLOCO 1: Carregamento do Dataset e Inspeção Inicial

In [ ]:
import pandas as pd
from google.colab import drive

# 1. Monta o Google Drive
drive.mount('/content/drive/')

# 2. Caminho da base de dados no Drive
path_f = '/content/drive/MyDrive/FABC/Base2FABC.xlsx'

# 3. Carrega a planilha Excel (aba 1)
df_fabc = pd.read_excel(path_f, sheet_name=1)

# 4. Inspeção inicial
print("Dimensões do dataset:", df_fabc.shape)
df_fabc.head()

## PILAR 2: ANÁLISE MATEMÁTICA E ALGORÍTMICA DETALHADA

### 2.1 A Função `df_quartile` (Cálculo Fino do Fatiamento)
1. **Tamanho do Quartil ($qsize$):**
   $$qsize = 1 + \lfloor N \times \alpha \rfloor$$
2. **Ordenação (`ascending`):**
   * `ascending=False`: Ordena do maior para o menor $\implies$ topo é a faixa de **Maior Produtividade (Best - B)**.
   * `ascending=True`: Ordena do menor para o maior $\implies$ topo é a faixa de **Menor Produtividade (Worst - W)**.
3. **`copy(deep=True)`**: Cria um espaço de memória novo desvinculado, evitando avisos do Pandas.

## BLOCO 2: Filtragem por Critério (Criação das Categorias B2C, W2C, B4C, W4C, B8C, W8C)

In [ ]:
def df_quartile(df, criterion, porc=0.25, quartile='first', ascending=False):
    qsize = 1 + int(df[df.columns[0]].count() * porc)
    dic = {
        'first': (0 * qsize, 1 * qsize - 1),
        'second': (1 * qsize, 2 * qsize - 1),
        'center': (int(1.5 * qsize), int(2.5 * qsize - 1))
    }
    dfsor = df.sort_values([criterion], ascending=[ascending])
    dfqua = pd.DataFrame(dfsor.iloc[dic[quartile][0]:dic[quartile][1]]).copy(deep=True)
    return dfqua

# Geração das fatias
B2C = df_quartile(df_fabc, 'Produtividade [kg/ha]', porc=0.50, quartile='first', ascending=False)
W2C = df_quartile(df_fabc, 'Produtividade [kg/ha]', porc=0.50, quartile='first', ascending=True)

B4C = df_quartile(df_fabc, 'Produtividade [kg/ha]', porc=0.25, quartile='first', ascending=False)
W4C = df_quartile(df_fabc, 'Produtividade [kg/ha]', porc=0.25, quartile='first', ascending=True)

B8C = df_quartile(df_fabc, 'Produtividade [kg/ha]', porc=0.125, quartile='first', ascending=False)
W8C = df_quartile(df_fabc, 'Produtividade [kg/ha]', porc=0.125, quartile='first', ascending=True)

print("Tamanhos gerados:")
print("B2C:", B2C.shape, "W2C:", W2C.shape)
print("B4C:", B4C.shape, "W4C:", W4C.shape)
print("B8C:", B8C.shape, "W8C:", W8C.shape)

## BLOCO 3: Visualização das Distribuições (Violin plot + Stripplot e KDE)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Adiciona rótulos para identificação
df_fabc['Label'] = 'df_fabc'
B2C['Label'] = 'B2C'
W2C['Label'] = 'W2C'
B4C['Label'] = 'B4C'
W4C['Label'] = 'W4C'
B8C['Label'] = 'B8C'
W8C['Label'] = 'W8C'

df_combined = pd.concat([df_fabc, B2C, W2C, B4C, W4C, B8C, W8C])

# Violim + Stripplot
plt.figure(figsize=(14, 6))
sns.violinplot(x='Label', y='Produtividade [kg/ha]', data=df_combined, palette='viridis', inner='point')
sns.stripplot(x='Label', y='Produtividade [kg/ha]', data=df_combined, color='k', alpha=0.5, size=2)
plt.title('Distribuição da Produtividade [kg/ha] entre Subconjuntos')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 2.2 Sanitização e Gravação de Atributos Individuais

O DAMICORE exige que cada atributo do dataset seja salvo como um arquivo de texto independente.
* **`unidecode`**: Transforma caracteres unicode (acentos, símbolos) em ASCII.
* **Limpeza de Nomes**: Remove caracteres especiais (`[`, `]`, `/`, `%`, etc.) que causam erros em sistemas POSIX ou bibliotecas C++.

## BLOCO 4: Sanitização e Exportação de Colunas Individuais

In [ ]:
import os
from unidecode import unidecode

b4c_dir = "/content/drive/MyDrive/FABC/Etapa13/data3/B4C"
os.makedirs(b4c_dir, exist_ok=True)

chars2replac = ['_', ' ', '[', ']', '/', '<', '>', '.', '%', '=', '(', ')']

for col in B4C.columns:
    if col == 'Label': continue
    fname = unidecode(str(col))
    for c in chars2replac:
        fname = fname.replace(c, '')
    full_path = os.path.join(b4c_dir, fname)
    with open(full_path, 'w') as f:
        B4C[col].to_csv(f)

print("Arquivos sanitizados salvos com sucesso.")

## PILAR 3: O MOTOR DAMICORE E A DISTÂNCIA DE COMPRESSÃO NORMALIZADA (NCD)

### 3.1 A Equação da NCD (Normalized Compression Distance)
$$NCD(x, y) = \frac{C(xy) - \min(C(x), C(y))}{\max(C(x), C(y))}$$

* $C(x)$: Tamanho em bytes do arquivo $x$ comprimido via `gzip`.
* $C(xy)$: Tamanho em bytes da concatenação dos arquivos $x$ e $y$ comprimidos via `gzip`.
* **$NCD(x, y) \approx 0$**: Variáveis redundantes ou fortemente correlacionadas.
* **$NCD(x, y) \approx 1$**: Variáveis independentes/sem relação de informação.

### 3.2 Construção do Filograma (Neighbor-Joining)
Emparelha os nós com menor $NCD$ criando uma **Árvore de Distância Hierárquica sem Raiz** (*Unrooted Tree*).

## BLOCO 5: Instalação e Execução Otimizada do DAMICORE

> **Nota de Engenharia de Computação:** Para evitar gargalos de E/S da API do Google Drive, os arquivos são copiados para o SSD local do Colab (`/content/`), processados em alta velocidade e os resultados são copiados de volta para o Drive.

In [ ]:
import shutil
import sys

# Instalação das dependências
!pip install -q git+https://gitlab.uspdigital.usp.br/jjuarez/damicore.git python-igraph

py_version = f"{sys.version_info.major}.{sys.version_info.minor}"
damicore_path = f'/usr/local/lib/python{py_version}/dist-packages/damicore/'
os.chdir(damicore_path)
import damicore as dm

# Otimização de E/S via SSD local
drive_input_dir = "/content/drive/MyDrive/FABC/Etapa13/data3/B4C"
drive_output_dir = "/content/drive/MyDrive/FABC/Etapa13/data3/B4Coutput"
local_input_dir = "/content/B4C_local"
local_output_dir = "/content/B4Coutput_local"

os.makedirs(local_output_dir, exist_ok=True)
if os.path.exists(local_input_dir):
    shutil.rmtree(local_input_dir)
shutil.copytree(drive_input_dir, local_input_dir)

argv = [
    "damicore.py", local_input_dir,
    "--compressor", "gzip",
    "--tree-output", os.path.join(local_output_dir, "1-tree.newick"),
    "--output", os.path.join(local_output_dir, "1-membership.csv"),
    "--partition-output", os.path.join(local_output_dir, "1-partition.csv"),
    "--ncd-output", os.path.join(local_output_dir, "1-ncd.csv"),
    "--community-detection", "fast"
]

dm.main(argv[1:])

# Copia os resultados de volta para o Google Drive
os.makedirs(drive_output_dir, exist_ok=True)
for item in os.listdir(local_output_dir):
    shutil.copy(os.path.join(local_output_dir, item), drive_output_dir)
print("Processamento DAMICORE concluído.")

## PILAR 4 & 5: ESTRUTURA DOS ARQUIVOS DE SAÍDA E CÁLCULO DE SENSIBILIDADE

### Estrutura dos Arquivos de Saída:
* `*-ncd.csv`: Matriz simétrica $NCD \in [0, 1]$.
* `*-tree.newick`: Estrutura topológica da árvore em notação Newick.
* `*-membership.csv`: ID de cada variável e o código do clado/comunidade.
* `*-partition.csv`: Métrica de modularidade da rede ($Q$-score).

### Cálculo da Distância Cofenética:
Calcula-se a soma dos comprimentos dos ramos entre a folha de cada variável e a folha `Produtividadekgha`. Quanto menor a distância NCD, maior a sensibilidade/influência da variável sobre a Produtividade.

## BLOCO 6: Leitura do Filograma e Cálculo do Ranking de Sensibilidade

In [ ]:
# Correção de compatibilidade para Python 3.13+ (PEP 594)
!pip install -q ete3 legacy-cgi
try:
    import cgi
except ImportError:
    import legacy_cgi as cgi
    sys.modules['cgi'] = cgi

from ete3 import Tree

# Carrega a árvore em formato Newick
tree_path = os.path.join(local_output_dir, "1-tree.newick")
t = Tree(tree_path, format=1)

# Calcula a distância cofenética (NCD) de todas as variáveis até a Produtividade
alvo_name = "Produtividadekgha"
no_alvo = t.search_nodes(name=alvo_name)[0]

ranking = []
for folha in t.get_leaves():
    if folha.name != alvo_name:
        dist = t.get_distance(no_alvo, folha)
        ranking.append({'Variavel': folha.name, 'Distancia_NCD': dist})

df_ranking = pd.DataFrame(ranking).sort_values(by='Distancia_NCD')
print("Top 10 Variáveis Mais Influentes/Sensíveis:")
df_ranking.head(10)